In [ ]:
from networkx.classes import non_edges
from nltk.translate.lepor import length_penalty
from pydantic.v1.utils import truncate
from rich.jupyter import display
from sympy.physics.units import temperature
! py -m pip install "transformers>=4.41" datasets accelerate peft evaluate rouge_score scikit-learn sentencepiece pandas scikit-learn matplotlib

In [ ]:
import os
from collections import Counter, OrderedDict
from dataclasses import dataclass
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
import evaluate
import numpy as np
import torch
from datasets import Dataset, load_dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    set_seed,
)

In [ ]:
import pandas as pd
from pathlib import Path
from os import getcwd
current_dir = getcwd()
LOCAL_DATA_PATH= Path(current_dir).parent / ".." / "datafiles" / "yelp_reviews_clean_CA.csv"
try:
    review_dataset = pd.read_csv(filepath_or_buffer=LOCAL_DATA_PATH,
                                 on_bad_lines="skip",
                                 encoding="utf-8",
                                 )
except FileNotFoundError as e:
    print(e)

training_data,validation_data = train_test_split(review_dataset, test_size=0.2, random_state=7,stratify=review_dataset['sentiment'])
training_data = training_data.to_dict(orient="records")
validation_data = validation_data.to_dict(orient="records")
def token_building_method(tokenizer, max_source_len, max_target_len):
   def builder_method(batch):
       inputs = tokenizer(
           batch["input"],
           max_length=max_source_len,
           truncation=True,
       )
       labels = tokenizer(
           text_target = batch["target"],
           truncation=True,
           max_length=max_target_len,
       )
       inputs["labels"] = [
           [(token_id if token_id != tokenizer.pad_token_id else -100) for token_id in seq]
           for seq in labels["input_ids"]
       ]
       return inputs
   return builder_method

In [ ]:
@dataclass
class FlanT5Config:
    model: str = "google/flan-t5-base"
    training_data:str = LOCAL_DATA_PATH
    eval_data:str = LOCAL_DATA_PATH
    output_dir:str = "./flant5-reviews"
    max_source_len: int = 512
    max_target_len: int = 128
    max_reviews_per_business: int = 30
    epochs: int = 1
    learning_rate: float = 5e-4
    batch_size:int = 16
    gradient_accumulation_steps: int = 8
    fine_tune_model: bool = False
    seed: int = 7
MODEL_ARGS = FlanT5Config()
MODEL_ARGS

### Instruction Definition Block


Since FLAN-T5 is an instruction based model, this block serves to define the instruction templates that will be used during training and infrencing. This section will also provide template response, since FLAN-T5 is an decoder-encoder style model there is no traditional classification, rather it is generating the sentiment which we will evaluate during training to see if they match

- `SENTIMENT_PROMPT` will be used to prompt the model to generate a "classification" that matches the provided reviews
- `REVIEW_SUMMARY_PROMPT` will be used to generate a short summary based on the provided reviews that will help inform users of our site with the general overview of other customer experiences. Highlighting any issues or concerns.

In [58]:
SENTIMENT_LABELS = ["negative", "neutral", "positive"]
SENTIMENT_PROMPT = (
    "classify the sentiment of this review, using negative, neutral, or positive. review: {input_review} \n sentiment class:"
)
REVIEW_SUMMARY_PROMPT = (
    "summarize the following reviews: {reviews}.\n"
    "business: {business}\n"
    "include the overall sentiment distribution provided by: {sentiment_distro}.\n"
    "Summary:"
)

In [ ]:
def label_dist(labels):
    labels = [tag for tag in labels if tag in SENTIMENT_LABELS]
    num_labels = len(labels)
    count = Counter(labels)
    result_percentage = {k: (round(100*count.get(k, 0)/num_labels) if num_labels else 0) for k in SENTIMENT_LABELS}
    return num_labels, result_percentage

def print_results(num_labels: int, result_percentage: dict) ->str:
    if num_labels == 0:
        return "No reviews with valid labels were detected"
    return(f"{num_labels}: reviews_detected, result_percentage_positive:{result_percentage['positive']} \n result_percentage_negative:{result_percentage['negative']} \n \
     results_precentage_neutral:{result_percentage['neutral']}  labels were detected")

def filter_reviews(reviews, tokenizer, review_limit, token_limit=256, review_length_limit=(MODEL_ARGS.max_source_len-len(REVIEW_SUMMARY_PROMPT))):
    filtered_output = []
    num_selected = 0
    for review in reviews[:review_limit]:
        review = (review or "").strip()
        if not review:
            continue
        ids = tokenizer(review, truncation=True, max_length=review_length_limit)['input_ids']
        if num_selected + len(ids) > token_limit and filtered_output:
            break
        filtered_output.append(tokenizer.decode(ids, skip_special_tokens=True).strip())
        num_selected += len(ids)
    return filtered_output

In [ ]:
def t5_multitask(rows, tokenizer=None, review_limit=30, review_length_limit=150):
    inputs = []
    targets = []
    tasks = []
    groups = OrderedDict()

    for row in rows:
        input = (row.get("text") or "").strip()
        if not input:
            continue
        label = (row.get("sentiment") or "").strip()
        if label in SENTIMENT_LABELS:
            inputs.append(SENTIMENT_PROMPT.format(input_review=input))
            targets.append(label)
            tasks.append("sentiment")

    return Dataset.from_dict({"input": inputs, "target": targets, "task": tasks})


In [65]:
@torch.inference_mode()
def sentiment_classifier(review, tokenizer, model, device, max_length= MODEL_ARGS.max_target_len):
    encoded = tokenizer(SENTIMENT_PROMPT.format(input_review=review), return_tensors="pt", truncate=True, max_lenth=max_length).to(device)
    output = model.generate(**encoded, max_new_tokens=5, num_beams=4)
    sentiment = tokenizer.decode(output[0], skip_special_tokens=True).strip().lower()
    return sentiment if sentiment in SENTIMENT_LABELS else "neutral"

@torch.inference_mode()
def business_review_summarizer(business_name, reviews,labels, model, tokenizer, device, max_reviews=20, max_source_len=MODEL_ARGS.max_source_len, review_length_limit=MODEL_ARGS.max_target_len):
    num_reviews, percentage = label_dist(labels)
    filtered_reviews = filter_reviews(reviews, tokenizer, review_limit=max_reviews, review_length_limit=review_length_limit)
    review_blob = "\n".join(f"-{text}" for text in filtered_reviews)
    summary_prompt = REVIEW_SUMMARY_PROMPT.format(
        business=business_name,
        reviews=review_blob,
        sentiment_distro=print_results(num_reviews, percentage),
    )
    encoded = tokenizer(summary_prompt, return_tensors="pt", truncation=True, max_length=max_source_len).to(device)
    output = model.generate(**encoded,max_new_tokens=review_length_limit,num_beams=1,no_repeat_ngram_size=3,length_penalty=2.5, repetition_penalty=2.0, do_sample=True, temperature=0.8)
    review_summary = tokenizer.decode(output[0], skip_special_tokens=True).strip()
    return {"business": business_name, "number_of_reviews":num_reviews, "sentiment_composition": percentage, "summary": review_summary}

In [ ]:
set_seed(MODEL_ARGS.seed)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device: ", device)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ARGS.model)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ARGS.model)
model.to(device)
print(f"Loaded: {MODEL_ARGS.model}")

In [62]:
# Zero-shot summarization on the freshly loaded base model (NO training yet).
from pprint import pprint

model.eval()
zeroshot_business = "Sample Cafe"
zeroshot_reviews = [
    "Great coffee and cozy seating, but service was slow at lunch.",
    "Overpriced pastries and the barista got my order wrong twice.",
    "Nice atmosphere but the tables were dirty and the music was too loud.",
    "Great food, and the ice coffee was tasty!",
    "Parking was a challenge but overall the experience was a nice one",
    "Lines were long, and it took forever to get my order"
]

print("ZERO-SHOT (base model, no fine-tuning):\n")
pprint(business_review_summarizer(
    zeroshot_business, zeroshot_reviews, model, tokenizer, device,
    max_reviews=MODEL_ARGS.max_reviews_per_business, max_source_len=MODEL_ARGS.max_source_len,
))
print(model.name_or_path)

NameError: name 'model' is not defined

In [ ]:
@dataclass
class Evaluation:
    tokenizer: object
    def __call__(self,eval_predictions):
        predictions, labels = eval_predictions
        if isinstance(predictions, tuple):
            predictions = predictions[0]
        predictions = np.where(predictions != -100, predictions, self.tokenizer.pad_token_id)
        labels = np.where(labels != -100, labels, self.tokenizer.pad_token_id)

        decoded_predictions = [prediction.strip() for prediction in self.tokenizer.batch_decode(predictions, skip_special_tokens=True)]
        decoded_labels = [label.strip() for label in self.tokenizer.batch_decode(labels, skip_special_tokens=True)]

        sentiment_index = [index for index, ref in enumerate(decoded_labels) if ref.lower() in SENTIMENT_LABELS]

        output = {}
        if sentiment_index:
            good_classification = sum(decoded_predictions[index].lower().startswith(decoded_labels[index].lower()) for index in sentiment_index)

            output["sentiment_classification_accuracy"] = accuracy_score(decoded_labels, decoded_predictions)
            precision, recall, f1, _ = precision_recall_fscore_support(decoded_labels, decoded_predictions,labels=SENTIMENT_LABELS,
                                                                       average="macro")
            output["sentiment_precision"] = precision
            output["sentiment_recall"] = recall
            output["sentiment_f1"] = f1
        return output

In [ ]:
assert os.path.exists(MODEL_ARGS.training_data),(
    f"training data not found"
)

if MODEL_ARGS.fine_tune_model:
    from peft import LoraConfig, TaskType, get_peft_model
    model = get_peft_model(
        model,
        LoraConfig(
            task_type=TaskType.SEQ_2_SEQ_LM,
            r=16, lora_alpha=32,lora_dropout=0.05,
            target_modules= ["q","v"],
        )
    )
    model.print_trainable_parameters()
    if MODEL_ARGS.learning_rate < 1e-3:
        print("learning rate too low for fine tuning of the model")
tokenizer_fn = token_building_method(tokenizer,MODEL_ARGS.max_source_len,MODEL_ARGS.max_target_len)
training_data = t5_multitask(training_data, tokenizer, MODEL_ARGS.max_reviews_per_business, MODEL_ARGS.max_source_len)
training_data = training_data.shuffle(seed=MODEL_ARGS.seed).map(tokenizer_fn, batched=True, remove_columns=training_data.column_names)

if MODEL_ARGS.eval_data and os.path.exists(MODEL_ARGS.eval_data):
    validation_data = t5_multitask(validation_data, tokenizer, MODEL_ARGS.max_reviews_per_business,MODEL_ARGS.max_source_len)
    validation_data = validation_data.map(tokenizer_fn, batched=True, remove_columns=validation_data.column_names)
sample_collator = DataCollatorForSeq2Seq(tokenizer, model=model, label_pad_token_id=-100,pad_to_multiple_of=8)

bf16_check = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

training_parameters = Seq2SeqTrainingArguments(
    output_dir=MODEL_ARGS.output_dir,
    per_device_train_batch_size=MODEL_ARGS.batch_size,
    per_device_eval_batch_size=MODEL_ARGS.batch_size,
    gradient_accumulation_steps=MODEL_ARGS.gradient_accumulation_steps,
    learning_rate=MODEL_ARGS.learning_rate,
    num_train_epochs=MODEL_ARGS.epochs,
    bf16=bf16_check,
    fp16=False,
    gradient_checkpointing=False,
    generation_max_length=MODEL_ARGS.max_target_len,
    eval_strategy="no",
    save_strategy="epoch",
    save_total_limit=2,
    logging_steps=20,
    report_to="none",
    dataloader_num_workers=8,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_parameters,
    train_dataset=training_data,
    eval_dataset=validation_data,
    data_collator=sample_collator,
)
print("entering train", flush=True)
trainer.train()
trainer.save_model(MODEL_ARGS.output_dir)
tokenizer.save_pretrained(MODEL_ARGS.output_dir)

In [ ]:
import os
import numpy as np
import torch
import evaluate
import json, datetime
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from os import getcwd
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from tqdm.auto import tqdm

GRAPHIC_DIR = os.path.join(MODEL_ARGS.output_dir, "eval_plots")
os.makedirs(GRAPHIC_DIR, exist_ok=True)
DPI = 150
device = "cuda" if torch.cuda.is_available() else "cpu"
CKPT = os.path.join(MODEL_ARGS.output_dir, "checkpoint-2171")

eval_tokenizer = AutoTokenizer.from_pretrained(CKPT)
eval_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ARGS.output_dir)

eval_model.to(device)
eval_model.eval()
print(f'Loaded model from {MODEL_ARGS.output_dir}')


current_dir = getcwd()
LOCAL_DATA_PATH= Path(current_dir).parent / ".." / "datafiles" / "yelp_reviews_clean_CA.csv"
try:
    review_dataset = pd.read_csv(filepath_or_buffer=LOCAL_DATA_PATH,
                                 on_bad_lines="skip",
                                 encoding="utf-8",
                                 )
except FileNotFoundError as e:
    print(e)

_,validation_data = train_test_split(review_dataset, test_size=0.2, random_state=7,stratify=review_dataset['sentiment'])
validation_data = validation_data.to_dict(orient="records")
display(validation_data)

EVALUATION_SENTIMENT_LIMIT = 3000

def evaluate_sentiment_classification(validation_data, eval_model, eval_tokenizer, labels=SENTIMENT_LABELS, eval_limit=EVALUATION_SENTIMENT_LIMIT):
    y_prediction, y_true = [], []
    for review in tqdm(validation_data, total=eval_limit, desc="Evaluating sentiment classification"):
        truth = (review.get("sentiment") or "").strip().lower()
        prediction = (review.get("text") or "").strip().lower()
        if truth not in labels or not prediction:
            continue
        y_true.append(truth)
        y_prediction.append(sentiment_classifier(prediction, eval_tokenizer, eval_model,device=device))
        if eval_limit and len(y_true) > eval_limit:
            break
    return y_prediction, y_true

y_prediction, y_true = evaluate_sentiment_classification(validation_data, eval_model, eval_tokenizer, labels=SENTIMENT_LABELS)

print("Sentiment Classification Report")
accuracy = accuracy_score(y_true, y_prediction)
macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(y_true, y_prediction,average='macro', labels=SENTIMENT_LABELS)
weighted_precision, weighted_recall,weighted_f1, _ = precision_recall_fscore_support(y_true, y_prediction,average='weighted', labels=SENTIMENT_LABELS)
confusion_mat = confusion_matrix(y_true, y_prediction, labels=SENTIMENT_LABELS)
print(f"accuracy : {accuracy:.3f}")
print(f"macro_precision : {macro_precision:.3f}. macro_recal : {macro_recall:.3f}, macro_f1 : {macro_f1:.3f}")
print(f"weighted_precision : {weighted_precision:.3f}, weighted_recal : {weighted_recall:.3f}, weighted_f1 : {weighted_f1:.3f}")

metrics = {
    "timestamp": datetime.datetime.now().isoformat(timespec="seconds"),
    "model": MODEL_ARGS.model,
    "n_examples": len(y_true),
    "accuracy": float(accuracy),
    "macro": {
        "precision": float(macro_precision),
        "recall": float(macro_recall),
        "f1": float(macro_f1),
    },
    "weighted": {
        "precision": float(weighted_precision),
        "recall": float(weighted_recall),
        "f1": float(weighted_f1),
    },
    "per_class": classification_report(
        y_true, y_prediction, labels=SENTIMENT_LABELS,
        output_dict=True, zero_division=0),
    "confusion_matrix": {
        "labels": SENTIMENT_LABELS,
        "matrix": confusion_mat.tolist(),   # rows=true, cols=pred
    },
}

metrics_path = os.path.join(GRAPHIC_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)

print("saved", metrics_path)

cm_figure, ax = plt.subplots(figsize=(5,4))
ConfusionMatrixDisplay(confusion_mat,display_labels=SENTIMENT_LABELS).plot(cmap="Blues", values_format="d", ax=ax, colorbar=False)
ax.set_title("Sentiment Classification Confusion Matrix: Flan-t5-base")
cm_figure.tight_layout()
path = os.path.join(GRAPHIC_DIR,"flan_t5_confusion_matrix.png")
cm_figure.savefig(path, dpi=DPI, bbox_inches="tight")
plt.show(); plt.close(cm_figure)

sentiment_report = classification_report(y_true, y_prediction, labels=SENTIMENT_LABELS, output_dict=True, zero_division=0)
report_df = pd.DataFrame(sentiment_report).T.reindex(SENTIMENT_LABELS)[["precision", "recall", "f1-score"]]
rep_fig, ax = plt.subplots(figsize=(6,4))
report_df.plot(kind="bar", ax=ax, ylim=(0,1),rot=0)
ax.set_title("Performance Metrics Per-Class")
ax.set_ylabel("score")
ax.legend(loc="lower right")
rep_fig.tight_layout()
path = os.path.join(GRAPHIC_DIR,"class_metric_report.png")
rep_fig.savefig(path, dpi=DPI, bbox_inches="tight")
plt.show(); plt.close(rep_fig)

try:
   validation_dataframe = pd.DataFrame(validation_data)
   date_col = next((col for col in validation_dataframe.columns if col.lower() in ("date", "review_date", "time")), None)
   if date_col and "sentiment" in validation_dataframe.columns:
       validation_dataframe[date_col] = pd.to_datetime(validation_dataframe[date_col], errors="coerce")
       validation_dataframe = validation_dataframe.dropna(subset=[date_col])
       month_plot = (validation_dataframe.set_index(date_col).groupby([pd.Grouper(freq="ME"), "sentiment"]).size().unstack(fill_value=0))
       if not month_plot.empty:
            fig, ax = plt.subplots(figsize=(8, 4))
            month_plot.plot(kind="area", stacked=True, ax=ax)
            ax.set_xlabel("date"); ax.set_ylabel("reviews")
            ax.set_title("Sentiment Over Time (gold labels)")
            fig.tight_layout()
            p = os.path.join(GRAPHIC_DIR, "sentiment_over_time.png")
            fig.savefig(p, dpi=DPI, bbox_inches="tight")
            plt.show(); plt.close(fig)
            print("saved", p)
       else:
            print("no month found")
except Exception as e:
    print(e)


In [72]:
import os
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
CKPT = os.path.join(MODEL_ARGS.output_dir, "checkpoint-2171")
KEEP_COLUMNS = ["business_name", "date", "sentiment"]
eval_tokenizer = AutoTokenizer.from_pretrained(CKPT)
eval_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ARGS.output_dir)

eval_model.to(device)
eval_model.eval()
print(f'Loaded model from {MODEL_ARGS.output_dir}')

@torch.inference_mode()
def classification_by_batch(reviews):
    sentiment_classification_prompt=[SENTIMENT_PROMPT.format(input_review=(review or "")) for review in reviews]
    encoded = eval_tokenizer(sentiment_classification_prompt, return_tensors="pt",padding=True,truncation=True,max_len=MODEL_ARGS.max_source_len).to(device)
    output = eval_model.generate(**encoded, max_new_tokens=5,num_beams=1)
    decoded = eval_tokenizer.batch_decode(output, skip_special_tokens=True)
    predictions = []
    for item in decoded:
        item = item.strip().lower()
        predictions.append(item if item in SENTIMENT_LABELS else "neutral")
    return predictions

def inference():
    review_dataframe = pd.DataFrame(review_dataset)
    texts = review_dataframe["text"].fillna("").tolist()
    INF_BATCH = 64
    predictions = []
    for i in tqdm(range(0, len(texts), INF_BATCH), desc="Inferring Flan-t5"):
        predictions.extend(classification_by_batch(texts[i:i+INF_BATCH]))
    review_dataframe["prediction_sentiment"] = predictions
    cols = [cols for cols in KEEP_COLUMNS if cols in review_dataframe.columns] + ["prediction_sentiment"]
    missing = [cols for cols in KEEP_COLUMNS if cols not in review_dataframe.columns]
    if missing:
        print(f"{len(missing)} columns missing")
    output_dataframe = review_dataframe[cols]
    output_dataframe.to_csv(os.path.join(MODEL_ARGS.output_dir, "flan_t5_sentiment.csv"), index=False)
    display(output_dataframe.head())

inference()

,business_name,date,sentiment,prediction_sentiment
0,Los Padres National Forest,2016-03-30,neutral,negative
1,Hibachi Steak House & Sushi Bar,2016-07-25,neutral,neutral
2,Sushi Teri,2013-09-04,positive,positive
3,The Original Habit Burger Grill,2017-01-02,positive,positive
4,Helena Avenue Bakery,2016-10-13,positive,positive


In [75]:
 MIN_REVIEWS = 30
device = "cuda" if torch.cuda.is_available() else "cpu"
eval_tokenizer = AutoTokenizer.from_pretrained(CKPT)
eval_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

eval_model.to(device)
eval_model.eval()
print(f'Loaded model from {MODEL_ARGS.output_dir}')
def summary_inf():
    # group reviews by business (single linear pass)
    groups = OrderedDict()
    for _, row in review_dataset.iterrows():
        business = str(row.get("business_name") or "").strip()
        review = str(row.get("text") or "").strip()
        if not business or not review:
            continue
        groups.setdefault(business, {"texts": [], "sentiments": []})
        groups[business]["texts"].append(review)
        groups[business]["sentiments"].append(str(row.get("sentiment") or "").strip().lower())

    # keep only businesses with enough reviews for a meaningful summary
    eligible = {b: v for b, v in groups.items() if len(v["texts"]) >= MIN_REVIEWS}
    print(f"{len(groups)} businesses total; {len(eligible)} with >= {MIN_REVIEWS} reviews")
    rows = []
    for business, v in tqdm(eligible.items(), desc="summarizing businesses"):
        summary_dict = business_review_summarizer(business,reviews=v["texts"],labels=v['sentiments'],model=eval_model, tokenizer=eval_tokenizer, device=device)
        rows.append({
            "business_name": summary_dict['business'],
            "n_reviews": summary_dict['number_of_reviews'],
            "sentiment_distro": summary_dict["sentiment_composition"],
            "summary": summary_dict['summary'],
        })

    out_df = pd.DataFrame(rows)
    out_df.to_csv(os.path.join(MODEL_ARGS.output_dir, "flan_t5_summary.csv"), index=False,na_rep="Missing", encoding="utf-8-sig")
    print(out_df.head())


summary_inf()